# Stage 1: LLM-Based Risk Extraction
### From Stage to System: Bias Propagation in Clinical AI Pipelines
**Luwam Major Kefali**

**Hilina Fissha Woreta**

This notebook builds Stage 1 of the pipeline: an open-weight LLM reads each patient's discharge summary and extracts structured risk information (comorbidity burden, social determinants of health, psychiatric complexity, discharge risk indicators) as JSON. This structured output feeds into Stage 2 (XGBoost readmission classifier) alongside the tabular features.

Everything here reads from `notes_index.parquet` and `mimic_features.parquet`, produced by the preprocessing notebook, and from `config.yaml` via the shared `config.py` loader, so nothing is hardcoded twice between our all the tracks.

## Setup

Installing the packages we need on top of what Kaggle ships by default. `bitsandbytes` lets us load the model in 4-bit, so that a 7B model comfortably fits on a single Kaggle GPU.

In [ ]:
 !pip install -q -U transformers accelerate bitsandbytes pydantic pyyaml outlines  

In [ ]:
# Import standard and third-party libraries
import os
import sys
import re
import json
import yaml
import time
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from pydantic import BaseModel, Field, ValidationError
from typing import List, Literal
import outlines

# Load configurations
config_dir = "/kaggle/input/datasets/luwammajor/clinical-pipeline-config"
if config_dir not in sys.path:
    sys.path.append(config_dir)

from config import load_config
cfg = load_config(config_path=f"{config_dir}/config.yaml")

# Load external prompt template
with open(f"{config_dir}/stage1_prompt.yaml", "r") as f:
    prompt_config = yaml.safe_load(f)
    
cfg.stage1.model_name = "meta-llama/Llama-3.1-8B-Instruct"

print("Environment and configuration loaded.")
print("active environment:", cfg.active_environment)
print("model:", cfg.stage1.model_name)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

# Retrieve the token from Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print("Hugging Face token loaded successfully from Kaggle Secrets.")
except Exception as e:
    print(f"Could not load Hugging Face token from secrets: {e}")

## Loading the notes

`notes_index.parquet` links each `hadm_id` to the ID of its latest discharge note, but not the note text itself. It was already filtered in preprocessing to only the admissions that survived the complete-labs cohort. We also pull `race_clean` and `readmitted_30d` from `mimic_features.parquet`, we need both for stratified sampling, before we touch any raw note text.

In [ ]:
notes_index = pd.read_parquet(cfg.paths.notes_index)
print("admissions with a discharge note available:", len(notes_index))

strat_cols = pd.read_parquet(
    cfg.paths.features, columns=["hadm_id"] + cfg.stage1.stratify_on
)

notes_index = notes_index.merge(strat_cols, on="hadm_id", how="left")
notes_index = notes_index.dropna(subset=cfg.stage1.stratify_on).reset_index(drop=True)
print("admissions with stratification labels available:", len(notes_index))

## Sampling the cohort for Stage 1

Running a 7B model over the full cohort is not realistic on Kaggle's GPU quota. Rough budget: at roughly 3 to 5 seconds per note (4-bit inference, up to 512 generated tokens), a cohort in the hundreds of thousands of admissions would take somewhere in the hundreds of GPU hours, against a weekly quota of about 30 hours and a total project budget of 100 hours each.

Instead we draw a stratified sample, stratified on `race_clean` and `readmitted_30d` by default (same convention used for the train/val/test split), so the sample stays representative on both the outcome and the protected attribute we care about most. `stage1.sample_size` in the config controls how many admissions actually get extracted, currently 4,000. At roughly 4 seconds a note that will be under 5 GPU hours, comfortably inside a single Kaggle session.

Rare strata (a race and outcome combination with fewer than 2 admissions) are kept in full rather than dropped, matching the handling in preprocessing.

This is an estimate, not a guarantee, the sanity check further down measures actual generation time on our hardware before we commit to the full run. If it comes in slower, we lower `stage1.sample_size` rather than push through the full run since it will not finish. 

Now join the sampled admissions against the raw note text. We filter to the sample before this join rather than after, `discharge.csv` holds every discharge note in MIMIC-IV-Note, there is no reason to carry the other roughly 99 percent of it in memory through the rest of the notebook.

## Section extraction

Discharge summaries average around 2,200 tokens, and most of that is not useful for risk extraction: sign-off boilerplate, medication reconciliation tables, deidentification placeholders. Feeding the whole note to the model wastes context and slows generation.

MIMIC discharge summaries follow a fairly consistent structure, with section headers as their own line ending in a colon (`Social History:`, `Brief Hospital Course:`, etc). We split on that pattern and keep only the sections we actually want, defined in `config.yaml` under `stage1.sections`.

In [ ]:
# Load dataset and merge features
notes_index = pd.read_parquet(cfg.paths.notes_index)
strat_cols = pd.read_parquet(
    cfg.paths.features, columns=["hadm_id"] + cfg.stage1.stratify_on
)

notes_index = notes_index.merge(strat_cols, on="hadm_id", how="left")
notes_index = notes_index.dropna(subset=cfg.stage1.stratify_on).reset_index(drop=True)

def stratified_sample(df, strat_cols, n_samples, random_state):
    """Draw a stratified sample, keeping rare strata in full."""
    strat_key = df[strat_cols].astype(str).agg("_".join, axis=1)
    strat_counts = strat_key.value_counts()
    valid = strat_counts[strat_counts >= 2].index
    rare = strat_counts[strat_counts < 2].index

    main = df[strat_key.isin(valid)].copy()
    leftover = df[strat_key.isin(rare)].copy()

    if n_samples >= len(main):
        sampled_main = main
    else:
        sampled_main, _ = train_test_split(
            main,
            train_size=n_samples,
            stratify=strat_key[strat_key.isin(valid)],
            random_state=random_state,
        )
    return pd.concat([sampled_main, leftover], ignore_index=True)

sampled_index = stratified_sample(
    notes_index,
    strat_cols=cfg.stage1.stratify_on,
    n_samples=cfg.stage1.sample_size,
    random_state=cfg.random_seed,
)

# Join sampled index with actual discharge texts
discharge = pd.read_csv(
    f"{cfg.paths.notes_dir}/discharge.csv",
    usecols=["note_id", "text"],
)
notes = sampled_index.merge(discharge, on="note_id", how="left")
notes = notes.dropna(subset=["text"]).reset_index(drop=True)

print(f"Total valid notes loaded for extraction: {len(notes)}")

In [ ]:
print("Stage 1 sample size:", len(sampled_index))
print("\nrace distribution, full cohort (%):")
print((notes_index["race_clean"].value_counts(normalize=True) * 100).round(1))
print("\nrace distribution, Stage 1 sample (%):")
print((sampled_index["race_clean"].value_counts(normalize=True) * 100).round(1))
print("\nreadmission rate, full cohort:  ", round(notes_index["readmitted_30d"].mean() * 100, 1), "%")
print("readmission rate, Stage 1 sample:", round(sampled_index["readmitted_30d"].mean() * 100, 1), "%")

## Loading the model

Using BioMistral-7B by default since it is ungated on Hugging Face and needs no extra setup on Kaggle. `meta-llama/Llama-3.1-8B-Instruct` is a reasonable alternative if we want to compare structured-output reliability later, it just needs an `HF_TOKEN` Kaggle secret and license acceptance first. Swapping models can be done by swapping the model definition line in `config.yaml`.

Loading in 4-bit via `bitsandbytes` so this comfortably fits a single Kaggle T4/P100.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import outlines

print("Initializing optimized 4-bit model...")

# Configure 4-bit quantization with float16 compute for faster processing on T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Enable sdpa attention implementation for native PyTorch acceleration
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
    torch_dtype=torch.float16,
)

# Wrap the model for Outlines structured generation
outlines_model = outlines.models.Transformers(model, tokenizer)
print("Model loaded successfully.")

### How much does section extraction actually save us

Sanity check on 50 random notes, comparing token count for the full note versus just the extracted sections.

## Structured output schema

The model has to return exactly these four fields, nothing else. `social_determinants_flags` uses a fixed vocabulary so we can explode it into binary columns later, the same pattern used for `cm_*` comorbidity flags in preprocessing. `discharge_risk_indicators` is deliberately free text since risk factors are too varied to force into a fixed category list, we keep the raw phrases for the report and the visualizer, and use the count as a numeric feature for Stage 2.

Using `pydantic` to validate the model's JSON actually matches this shape before we trust it.

In [ ]:
import os
import json
import pandas as pd
from typing import List
from pydantic import BaseModel, Field
import outlines

SDOH_CATEGORIES = [
    "housing_instability", "food_insecurity", "substance_use", 
    "limited_social_support", "unemployment", "transportation_barrier"
]

class RiskExtraction(BaseModel):
    comorbidity_burden_score: int = Field(ge=0, le=10)
    sdoh_evidence: str
    social_determinants_flags: List[str]
    psychiatric_complexity: str
    # INSTRUCTION: Enforce max 3 items strictly at the grammar level
    discharge_risk_indicators: List[str] = Field(max_length=3)

prompt_template = """<s>[INST] You are a strict clinical AI information extraction system. Read the clinical note and extract risk factors into a valid JSON object matching the schema.

SCHEMA FIELDS:
- comorbidity_burden_score: Integer (0-10), 1 point per chronic condition explicitly mentioned.
- sdoh_evidence: Short quote (max 15 words) proving the flags, or "no evidence" if none exist.
- social_determinants_flags: Array of strings chosen ONLY from: housing_instability, food_insecurity, substance_use, limited_social_support, unemployment, transportation_barrier. If sdoh_evidence is "no evidence", this MUST be [].
- psychiatric_complexity: Classify strictly based on the text as "low" (none/baseline), "medium" (managed anxiety/depression), or "high" (severe/active psychosis, bipolar, schizophrenia).
- discharge_risk_indicators: Array of max 3 short phrases extracted directly from the text representing primary diagnoses or complications. If none, output [].

RULES:
1. Base all extractions strictly on the clinical text provided. Do not assume or invent facts.
2. Medical conditions are NEVER social determinants.
3. If no social determinants are found, output "no evidence" and [].
4. Exact Mapping: The flags MUST match the evidence exactly. If evidence is "substance use", output ONLY ["substance_use"]. Do NOT infer housing instability, unemployment, or poor support from substance use.
5. Cognitive impairment (e.g., dementia, delirium, Alzheimer's, "poor historian") is a medical/neurological issue. Do NOT flag it as a social determinant or as psychiatric complexity.

CLINICAL NOTE:
{clinical_note_text}

Extract the JSON object now: [/INST]"""

# INSTRUCTION: Cut context to 6000 characters to ensure sub-12 hour runtime on Kaggle T4
def smart_chunk_text(raw_note: str, max_chars: int = 6000) -> str:
    if not isinstance(raw_note, str):
        return ""
    if len(raw_note) <= max_chars:
        return raw_note
    head = raw_note[:3000]
    tail = raw_note[-3000:]
    return f"{head}\n\n...[TEXT TRUNCATED FOR LENGTH]...\n\n{tail}"

def build_prompt(clinical_text: str) -> str:
    chunked_text = smart_chunk_text(clinical_text)
    return prompt_template.format(clinical_note_text=chunked_text)

## Generation

`do_sample=False` gives greedy, deterministic decoding, so `temperature` is only passed when sampling is actually on. Truncating input to 4096 tokens as a hard safety cap, section extraction should already keep us well under that.

In [ ]:
def generate_extraction(note_sections: str) -> RiskExtraction:
    """Passes the chunked prompt to the constrained decoder for a single note."""
    prompt = build_prompt(note_sections)
    
    try:
        # Using 1024 to completely eliminate EOF cut-offs during single-note testing
        result = outlines_model(prompt, RiskExtraction, max_new_tokens=1024)
    except TypeError:
        # Generator fallback
        generator = outlines.generate.json(outlines_model, RiskExtraction)
        result = generator(prompt, max_tokens=1024)
    
    # Outlines guarantees a valid JSON string, which we safely load into our Python object
    if isinstance(result, str):
        return RiskExtraction.model_validate_json(result)
    return result

## Sanity check on a handful of notes

Before running this on the full sample, we look at five real outputs by eye, and time them. So that problems are caught earlier instead of after committing several GPU hours to a  batch run. 

### We first check with a synthetic "gold standard" note that we know contains severe social and psychiatric markers to test whether the extractor works as intended. 

In [ ]:
# Set the ID to extract
KNOWN_HADM_ID = 26131656

# Identify the correct column
text_col = 'text' if 'text' in notes.columns else 'sections_extracted'

# Safely extract the exact raw text
matched_row = notes[(notes['hadm_id'] == KNOWN_HADM_ID) | (notes['hadm_id'] == str(KNOWN_HADM_ID))]

if len(matched_row) > 0:
    raw_text = matched_row.iloc[0][text_col]
    
    print(f"=== FULL TEXT FOR HADM_ID: {KNOWN_HADM_ID} ===\n")
    # Print the full text exactly as it appears in the dataset
    print(raw_text)
    print("\n=== END OF TEXT ===")
else:
    print(f"ID {KNOWN_HADM_ID} not found.")

#### Inspect a discharge note text

In [ ]:
import time
import torch

# Replace with the actual ID (e.g., 28420282)
KNOWN_HADM_ID = 26131656

# 
text_col = 'text' if 'text' in notes.columns else 'sections_extracted'

# Fetch the exact row
target_row = notes[notes['hadm_id'] == KNOWN_HADM_ID].iloc[0]
print(f"Testing pipeline for hadm_id: {KNOWN_HADM_ID}")

with torch.inference_mode():
    start_time = time.time()
    
    # Apply the smart chunking and prompt template
    prompt = build_prompt(target_row[text_col])
    
    # Extract using the strict 120 token limit
    record = outlines_model(prompt, RiskExtraction, max_new_tokens=120)
    
    if isinstance(record, str):
        record = RiskExtraction.model_validate_json(record)
        
    elapsed_time = time.time() - start_time
    
    print(f"Extraction completed in {elapsed_time:.2f} seconds.")
    print("\n=== PIPELINE OUTPUT ===")
    print(record.model_dump())

In [ ]:
import time

print("Running 8-note verification...\n")

sample_notes = notes.sample(8, random_state=42)
text_col = 'text' if 'text' in sample_notes.columns else 'sections_extracted'
total_time = 0

with torch.inference_mode():
    for row in sample_notes.itertuples():
        start_time = time.time()
        prompt = build_prompt(getattr(row, text_col))
        
        try:
            record = outlines_model(prompt, RiskExtraction, max_new_tokens=150)
            if isinstance(record, str):
                record = RiskExtraction.model_validate_json(record)
                
            elapsed = time.time() - start_time
            total_time += elapsed
            
            print(f"hadm_id: {row.hadm_id} | Time: {elapsed:.2f}s | SUCCESS")
            print(record.model_dump())
        except Exception as e:
            print(f"hadm_id: {row.hadm_id} | FAILED: {str(e)}")
        print("-" * 80)

avg_time = total_time / len(sample_notes)
print(f"\nAverage generation time: {avg_time:.2f}s per note")
print(f"Projected time for 4,000 notes: {(avg_time * 4000) / 3600:.2f} hours")

Check both the outputs and the projected runtime above before continuing. If `parsed` is `None` for most of them, the prompt or schema needs adjusting, not the batch loop below. Common failure modes: the model adds explanatory text before the JSON, or invents a field name that does not match `RiskExtraction`. The `extract_json_block` regex handles the first case, `pydantic` validation catches the second and reports it as a parse failure rather than silently accepting bad data.

If the projected runtime is too long, lower `stage1.sample_size` in the config now, before the batch run below, rather than stopping it partway through.

## Full batch run

Checkpointing every `stage1.checkpoint_every` notes, so a Kaggle session timeout does not lose everything, and the loop resumes automatically from the last checkpoint on rerun.

In [ ]:
import os
import json
import pandas as pd
import torch
import gc

#  Read from the uploaded dataset, write to the working directory
input_checkpoint = "/kaggle/input/datasets/luwammajor/stage1-checkpoint/stage1_output.parquet"
output_checkpoint = "/kaggle/working/stage1_output.parquet"
error_log_path = "/kaggle/working/stage1_error_log.jsonl"

def run_stage1_fast_sequential(notes_df, input_checkpoint, output_checkpoint, error_log_path, checkpoint_every=100):
    results = []
    processed_ids = set()

    # Load the notes  already processed
    if os.path.exists(input_checkpoint):
        existing = pd.read_parquet(input_checkpoint)
        results = existing.to_dict("records")
        processed_ids = set(existing["hadm_id"])
        print(f"Resuming from checkpoint: {len(processed_ids)} already processed")
        
    # Also check the working directory in case the run crashes and needs to restart locally
    elif os.path.exists(output_checkpoint):
        existing = pd.read_parquet(output_checkpoint)
        results = existing.to_dict("records")
        processed_ids = set(existing["hadm_id"])
        print(f"Resuming from local working checkpoint: {len(processed_ids)} already processed")

    remaining = notes_df[~notes_df["hadm_id"].isin(processed_ids)]
    print(f"Notes remaining to process: {len(remaining)}")
    
    text_col = 'text' if 'text' in remaining.columns else 'sections_extracted'

    with torch.inference_mode():
        for i, row in enumerate(remaining.itertuples(), start=1):
            try:
                prompt = build_prompt(getattr(row, text_col))
                record = outlines_model(prompt, RiskExtraction, max_new_tokens=150)
                
                if isinstance(record, str):
                    record = RiskExtraction.model_validate_json(record)
                    
                results.append(explode_flags(record, row.hadm_id))

            except Exception as e:
                with open(error_log_path, "a") as f:
                    f.write(json.dumps({"hadm_id": row.hadm_id, "error": str(e)}) + "\n")
                
                results.append({
                    "hadm_id": row.hadm_id,
                    "extraction_success": False,
                    "plausible": False,
                    "comorbidity_burden_score": 0,
                    "psychiatric_complexity": "low",
                    "sdoh_evidence": "ERROR",
                    "discharge_risk_indicator_count": 0,
                    "discharge_risk_indicators_text": "",
                    **{f"sdoh_{cat}": 0 for cat in SDOH_CATEGORIES}
                })

            if i % checkpoint_every == 0 or i == len(remaining):
                pd.DataFrame(results).to_parquet(output_checkpoint, index=False)
                print(f"Progress: {i}/{len(remaining)} notes processed.")
                
                # Force garbage collection to prevent memory leak
                torch.cuda.empty_cache()
                gc.collect()

    final_df = pd.DataFrame(results)
    final_df.to_parquet(output_checkpoint, index=False)
    print("SUCCESS: Extraction complete!")
    return final_df

stage1_raw = run_stage1_fast_sequential(
    notes_df=notes, 
    input_checkpoint=input_checkpoint,
    output_checkpoint=output_checkpoint,
    error_log_path=error_log_path,
    checkpoint_every=100
)

## Extraction Reliability and Fairness Audit

Before passing the structured JSON artifacts to the readmission classifier, we must validate the integrity of the extraction layer. In the context of the bias migration hypothesis, algorithmic disparities can manifest as early as the natural language processing stage. 

The following evaluation performs two functions:
1. Calculates overall extraction reliability to ensure the prompt constraints held during the batch run.
2. Audits the success and plausibility rates across protected demographic groups (e.g., race). Variations in these metrics indicate bias introduced during Stage 1.

In [ ]:
import pandas as pd
from IPython.display import display

# 1. Calculate base counts
total_sampled = len(stage1_raw)
success_count = int(stage1_raw["extraction_success"].sum())
failed_count = int((~stage1_raw["extraction_success"]).sum())

plausible_count = int(stage1_raw.loc[stage1_raw["extraction_success"], "plausible"].sum())
implausible_count = int((~stage1_raw.loc[stage1_raw["extraction_success"], "plausible"]).sum())

# 2. Build the dataframe with accurate conditional percentages
summary_table = pd.DataFrame([
    {"metric": "total sampled", "count": total_sampled, "percent": 100.0},
    {"metric": "extraction succeeded", "count": success_count, "percent": round((success_count / total_sampled) * 100, 1)},
    {"metric": "extraction failed", "count": failed_count, "percent": round((failed_count / total_sampled) * 100, 1)},
    {"metric": "plausible (of succeeded)", "count": plausible_count, "percent": round((plausible_count / max(1, success_count)) * 100, 1)},
    {"metric": "flagged implausible (of succeeded)", "count": implausible_count, "percent": round((implausible_count / max(1, success_count)) * 100, 1)},
]).set_index("metric")

print("Pipeline Stage 1: Overall Extraction Reliability")
display(summary_table)

# Evaluate extraction reliability across protected demographic groups.
# Document any subgroup variance, as this represents early pipeline bias.
reliability_by_race = stage1_raw.merge(
    notes[["hadm_id", "race_clean"]], on="hadm_id", how="left"
).groupby("race_clean").agg(
    n=("hadm_id", "count"),
    success_rate=("extraction_success", "mean"),
    plausible_rate=("plausible", "mean"),
).round(3)

reliability_by_race["success_rate"] *= 100
reliability_by_race["plausible_rate"] *= 100
print("\nPipeline Stage 1: Extraction Reliability by Race")
display(reliability_by_race)

### Extracted Feature Distributions

Visualizing the distribution of the structured outputs confirms that the model is appropriately scaling risk rather than collapsing to binary extremes. Suspiciously flat or bimodal distributions suggest prompt compliance failure, which must be addressed prior to Stage 2 training. The generated figure is saved automatically for inclusion in the final LaTeX artifact.

In [ ]:
import matplotlib.pyplot as plt
import os

# Filter to include only successfully extracted and plausible records.
successful = stage1_raw[stage1_raw["extraction_success"] & stage1_raw["plausible"]]

fig, axes = plt.subplots(2, 2, figsize=(11, 8))

# Plot comorbidity burden distributions to verify appropriate variance.
axes[0, 0].hist(successful["comorbidity_burden_score"], bins=11, range=(0, 10), color="#4C72B0", edgecolor="white")
axes[0, 0].set_title("Comorbidity burden score")
axes[0, 0].set_xlabel("score (0-10)")

# Plot psychiatric complexity classifications.
successful["psychiatric_complexity"].value_counts().reindex(["low", "medium", "high"]).plot(
    kind="bar", ax=axes[0, 1], color="#55A868"
)
axes[0, 1].set_title("Psychiatric complexity")
axes[0, 1].tick_params(axis="x", rotation=0)

# Plot the incidence rates of distinct social determinants.
# INSTRUCTION: Ensure SDOH_CATEGORIES is defined in a previous cell
sdoh_cols = [f"sdoh_{c}" for c in SDOH_CATEGORIES]
successful[sdoh_cols].mean().sort_values().plot(kind="barh", ax=axes[1, 0], color="#C44E52")
axes[1, 0].set_title("SDOH flag rate")
axes[1, 0].set_xlabel("fraction of admissions flagged")

# Plot the volume of readmission risk factors extracted per patient.
successful["discharge_risk_indicator_count"].value_counts().sort_index().plot(
    kind="bar", ax=axes[1, 1], color="#8172B2"
)
axes[1, 1].set_title("Discharge risk indicator count")
axes[1, 1].tick_params(axis="x", rotation=0)

plt.tight_layout()

# INSTRUCTION: Safely resolve the output directory to prevent NameError in Kaggle
try:
    output_dir = cfg.paths.output_dir
except NameError:
    output_dir = "/kaggle/working"

# Save the figure directly to the configured output directory for reporting.
os.makedirs(output_dir, exist_ok=True)
plot_path = f"{output_dir}/stage1_field_distributions.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")

print(f"Distribution plots saved to: {plot_path}")
plt.show()

## Saving the output

This is what Stage 2 will merge in on `hadm_id`, alongside the tabular features from preprocessing.

In [ ]:
stage1_raw.to_parquet(cfg.paths.stage1_output, index=False)

# Verify the saved output length and structure
print(f"saved: {cfg.paths.stage1_output}")
print(f"rows: {len(stage1_raw)}")
print(stage1_raw.head())

## Manual evaluation sample

We don't have ground truth for any of this yet. Our report commits to validating Stage 1 against manually annotated notes, so this samples a set of extractions and writes a CSV with blank columns for us to fill in by hand. This is what the evaluation metric in the report (precision and recall of extracted fields against manual annotations) will actually be computed from.

In [ ]:
import pandas as pd

annotation_cols = [
    "human_comorbidity_score",
    "human_sdoh_evidence", 
    "human_sdoh_flags",
    "human_psychiatric_complexity",
    "human_discharge_risk_indicators"
]

# Sample 100 random notes using a fixed seed for strict reproducibility
manual_sample = notes.sample(n=100, random_state=42).copy()

# Initialize the empty annotation columns in the dataframe
for col in annotation_cols:
    manual_sample[col] = ""

# Select the text column dynamically 
text_col = 'text' if 'text' in manual_sample.columns else 'sections_extracted'

# Order the final columns for the CSV export
export_cols = ['hadm_id', text_col] + annotation_cols
manual_sample_export = manual_sample[export_cols]

# Export to a CSV file in the Kaggle 
output_csv_path = "/kaggle/working/manual_annotation_sample_100.csv"
manual_sample_export.to_csv(output_csv_path, index=False)

print(f"Successfully generated 100-note annotation sample at: {output_csv_path}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# Load Stage 1 data and base notes
stage1_df = pd.read_parquet("/kaggle/working/stage1_output.parquet")
successful = stage1_df[stage1_df["extraction_success"] == True].copy()

# Instruction: Ensure your original notes dataframe is available to calculate text length
# Assuming 'notes' is already loaded in your environment
text_col = 'text' if 'text' in notes.columns else 'sections_extracted'
notes['note_length'] = notes[text_col].astype(str).apply(len)
successful = successful.merge(notes[['hadm_id', 'note_length']], on='hadm_id', how='left')

fig = plt.figure(figsize=(16, 10))
plt.rcParams.update({'font.size': 12})

# 1. Truncation Bias Assessment (Scatter/Hexbin)
ax1 = plt.subplot(2, 2, 1)
hb = ax1.hexbin(successful['note_length'], successful['discharge_risk_indicator_count'], 
                gridsize=30, cmap='Blues', mincnt=1)
cb = fig.colorbar(hb, ax=ax1, label='Number of Patients')
ax1.set_title("Truncation Bias Assessment: Note Length vs. Extraction Count")
ax1.set_xlabel("Original Clinical Note Length (Characters)")
ax1.set_ylabel("Discharge Risk Indicators Extracted")
ax1.axvline(x=6000, color='red', linestyle='--', label='6,000 Char Cutoff')
ax1.legend()

# 2. Extracted Phrase Grounding (N-grams)
ax2 = plt.subplot(2, 2, 2)
# Instruction: Filter out empty text to avoid vectorizer errors
valid_text = successful[successful['discharge_risk_indicators_text'] != '']['discharge_risk_indicators_text']
vectorizer = CountVectorizer(ngram_range=(2, 3), stop_words='english', max_features=15)
X = vectorizer.fit_transform(valid_text)
ngrams = vectorizer.get_feature_names_out()
counts = X.sum(axis=0).A1
ngram_df = pd.DataFrame({'ngram': ngrams, 'count': counts}).sort_values(by='count', ascending=True)
ax2.barh(ngram_df['ngram'], ngram_df['count'], color='#4C72B0')
ax2.set_title("Top 15 Extracted Clinical Bigrams/Trigrams")
ax2.set_xlabel("Frequency in Dataset")

# 3. Feature Co-occurrence Matrix
ax3 = plt.subplot(2, 1, 2)
# Instruction: Map psychiatric complexity to numeric severity for correlation
psych_map = {'low': 0, 'medium': 1, 'high': 2}
successful['psych_numeric'] = successful['psychiatric_complexity'].map(psych_map)

sdoh_cols = [col for col in successful.columns if col.startswith('sdoh_') and col != 'sdoh_evidence']
matrix_cols = ['comorbidity_burden_score', 'psych_numeric'] + sdoh_cols

# Instruction: Calculate correlation matrix
corr_matrix = successful[matrix_cols].corr(method='spearman')
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, center=0, 
            square=True, linewidths=.5, cbar_kws={"shrink": .5}, ax=ax3)
ax3.set_title("Feature Co-occurrence and Independence Matrix")
ax3.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("/kaggle/working/stage1_internal_validation.png", dpi=300, bbox_inches="tight")
plt.show()